# Data exploration

Class distribution and sample images for both tasks.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from src.config import TASKS


## Class counts

In [ ]:
def count_split(path):
    return {d.name: sum(1 for f in d.iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png'})
            for d in sorted(path.iterdir()) if d.is_dir()}

rows = []
for task_name, task in TASKS.items():
    for split, path in [('train', task.train_dir), ('val', task.val_dir), ('test', task.test_dir)]:
        for cls, n in count_split(path).items():
            rows.append({'task': task_name, 'split': split, 'class': cls, 'count': n})
    for name, path in task.extra_test_dirs.items():
        for cls, n in count_split(path).items():
            rows.append({'task': task_name, 'split': name, 'class': cls, 'count': n})

counts = pd.DataFrame(rows)
counts.pivot_table(index=['task', 'class'], columns='split', values='count', fill_value=0)


## Sample images per class

In [ ]:
def show_samples(task, n=4):
    classes = sorted(d for d in task.train_dir.iterdir() if d.is_dir())
    fig, axes = plt.subplots(len(classes), n, figsize=(2.2 * n, 2.2 * len(classes)))
    if len(classes) == 1:
        axes = axes[None, :]
    for i, cls in enumerate(classes):
        files = [f for f in cls.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}][:n]
        for j in range(n):
            ax = axes[i, j]
            ax.axis('off')
            if j < len(files):
                ax.imshow(Image.open(files[j]).resize((160, 160)))
            if j == 0:
                ax.set_title(cls.name, fontsize=10, loc='left')
    fig.tight_layout()
    return fig

for name, task in TASKS.items():
    print(name)
    show_samples(task)
    plt.show()
